# W6Q3 — Digital Campaign ROI
**Question:** Which paid campaigns are most efficient? What is the cost per conversion, and where is budget being wasted?

**Audience:** Marketing & Leadership  
**Data source:** `ANALYTICS.MARTS.MART_CAMPAIGN_ANALYTICS`  
**SQL:** `sql/W6Q3_digital_campaign_roi.sql`

---

> ⚠️ **Double-counting risk:** `stg_campaign_performance` and `stg_campaign_cost` are separate
> staging tables contributing to `mart_campaign_analytics`. Verify that `daily_cost` does not
> double-count spend before trusting totals. Flag in findings if costs appear anomalously high.

**ROI metrics:**
| Metric | Definition |
|--------|-----------|
| CTR | clicks / impressions |
| CPC | cost / clicks |
| Cost per conversion | cost / total conversions |

## Setup

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

from src.connection import query

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 130})
PALETTE = sns.color_palette('muted', 8)

## Load Data

In [ ]:
with open('../sql/W6Q3_digital_campaign_roi.sql') as f:
    sql = f.read()

df = query(sql)
df['campaign_start'] = pd.to_datetime(df['campaign_start'])
df['campaign_end']   = pd.to_datetime(df['campaign_end'])
print(f"Campaigns: {len(df)}")
print(f"Total spend: ${df['total_cost'].sum():,.2f}")
print(f"Total conversions: {df['total_conversions'].sum():,}")
display(df.sort_values('total_cost', ascending=False))

## Cost per Conversion by Campaign

In [ ]:
df_sorted = df.sort_values('overall_cost_per_conversion')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].barh(df_sorted['campaign_name'], df_sorted['overall_cost_per_conversion'], color=PALETTE[0])
axes[0].set_title('Cost per Conversion by Campaign', fontsize=12)
axes[0].set_xlabel('$ Cost per Conversion')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
sns.despine(ax=axes[0])

df_ctr = df.sort_values('overall_ctr', ascending=False)
axes[1].barh(df_ctr['campaign_name'], df_ctr['overall_ctr'], color=PALETTE[1])
axes[1].set_title('Click-Through Rate (CTR) by Campaign', fontsize=12)
axes[1].set_xlabel('CTR (%)')
sns.despine(ax=axes[1])

fig.suptitle('Campaign Efficiency Metrics', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../output/W6Q3_campaign_efficiency.png', bbox_inches='tight')
plt.show()

## Spend vs Conversions — Bubble Chart

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sc = ax.scatter(df['total_cost'], df['total_conversions'],
                s=df['total_impressions'] / df['total_impressions'].max() * 2000,
                alpha=0.7, color=PALETTE[0])
for _, row in df.iterrows():
    ax.annotate(row['campaign_name'], (row['total_cost'], row['total_conversions']),
                textcoords='offset points', xytext=(6, 3), fontsize=8)
ax.set_title('Total Spend vs Conversions (bubble size = impressions)', fontsize=13, pad=12)
ax.set_xlabel('Total Spend ($)')
ax.set_ylabel('Total Conversions')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
sns.despine()
plt.tight_layout()
plt.savefig('../output/W6Q3_spend_vs_conversions.png', bbox_inches='tight')
plt.show()

## Findings

- **⚠️ Double-counting check (verify before trusting spend):** ...
- **Total spend across all campaigns:** ...
- **Most efficient campaign (lowest cost per conversion):** ...
- **Least efficient campaign:** ...
- **Highest CTR campaign:** ...
- **Active vs inactive campaign performance:** ...
- **Recommendation:** ...